# Analysis tables

Reads `results_csv/results.csv` (written by `load.ipynb`). Each table goes to `results_xlsx/<name>.xlsx` with
`lib.to_xlsx(name, table, pivot)`: sheet `table` (long form) and, when given, sheet `pivot`.

Metrics: **optimum** (proven optimal, exit 30: count), **win** (best cost among the solvers: count),
**score** (cost / best cost of the instance, 0 without a model: mean).

In [1]:
%load_ext autoreload
%autoreload 2
import pandas as pd
import lib
pd.set_option("display.max_columns", 50)

df = pd.read_csv(lib.CSV_DIR / "results.csv", dtype={"family": str})
SAVE = True                       # write the xlsx files
saved = []
def save(name, table, pivot=None):
    if SAVE:
        saved.append(lib.to_xlsx(name, table, pivot))
print(len(df), "runs,", df["solver"].nunique(), "solvers:", sorted(df["solver"].unique()))

3080 runs, 4 solvers: ['amo-ijca', 'amo-minf-lazy', 'amo-nomi-lazy', 'clingo']


## Per benchmark

In [2]:
table = lib.metrics_table(df, ["bench"])
pivot = lib.metrics_pivot(df, ["bench"])
save("by_benchmark", table, pivot)
pivot

bench              CA                   K                TSP             \
metric        optimum  win  score optimum win  score optimum win  score   
solver                                                                    
clingo            109  151  0.999       0   9  0.965       2   9  0.819   
amo-ijca          167  168  1.000       0   1  0.975       5  14  0.923   
amo-minf-lazy     169  168  1.000       0   1  0.959       4   4  0.887   
amo-nomi-lazy     167  168  1.000       0  89  0.998       4  83  0.996   

bench             WGC                 WOD              Total              
metric        optimum  win  score optimum win  score optimum  win  score  
solver                                                                    
clingo              0    0  0.754       0   0  0.122     111  169  0.762  
amo-ijca            0   20  0.937      24  24  0.317     196  227  0.873  
amo-minf-lazy       0  245  0.996      50  50  0.540     223  468  0.919  
amo-nomi-lazy       0   35  0.946      52  55  0.550     223  430  0.920

## Per benchmark and family

In [3]:
table = lib.metrics_table(df, ["bench", "family"])
pivot = lib.metrics_pivot(df, ["bench", "family"], rows="by")
save("by_family", table, pivot)
pivot

solver        clingo             amo-ijca             amo-minf-lazy       \
metric       optimum  win  score  optimum  win  score       optimum  win   
bench family                                                               
CA    paths       42   67  0.998       83   84  1.000            85   84   
      sched       67   84  1.000       84   84  1.000            84   84   
K     type4        0    9  0.965        0    1  0.975             0    1   
TSP   -            2    9  0.819        5   14  0.923             4    4   
WGC   125          0    0  0.794        0   10  0.963             0   25   
      130          0    0  0.773        0    5  0.955             0   35   
      135          0    0  0.748        0    5  0.924             0   40   
      140          0    0  0.741        0    0  0.914             0   50   
      145          0    0  0.726        0    0  0.921             0   45   
      150          0    0  0.743        0    0  0.945             0   50   
WOD   0_2          0    0  0.185       10   10  0.425            17   17   
      0_5          0    0  0.159       10   10  0.410            19   19   
      0_7          0    0  0.052        4    4  0.173            14   14   
Total            111  169  0.762      196  227  0.873           223  468   

solver              amo-nomi-lazy              
metric        score       optimum  win  score  
bench family                                   
CA    paths   1.000            83   84  1.000  
      sched   1.000            84   84  1.000  
K     type4   0.959             0   89  0.998  
TSP   -       0.887             4   83  0.996  
WGC   125     0.989             0   15  0.975  
      130     0.993             0   10  0.963  
      135     0.995             0    5  0.946  
      140     1.000             0    0  0.933  
      145     0.999             0    5  0.927  
      150     1.000             0    0  0.936  
WOD   0_2     0.630            17   17  0.630  
      0_5     0.650            20   21  0.656  
      0_7     0.394            15   17  0.415  
Total         0.919           223  430  0.920

## Overall

In [4]:
table = lib.metrics_table(df, [])
save("overall", table)
table

,solver,instances,solved,optimum,win,score
0,clingo,770,725,111,169,0.762
1,amo-ijca,770,725,196,227,0.873
2,amo-minf-lazy,770,725,223,468,0.919
3,amo-nomi-lazy,770,725,223,430,0.920


## How the runs ended

In [5]:
table = df.groupby(["bench", "solver", "status"]).size().reset_index(name="runs")
pivot = lib.status_pivot(df, ["bench"])
save("status", table, pivot)
pivot

bench               CA                       K                 TSP             \
status        complete outof time outof memory outof time complete outof time   
solver                                                                          
clingo             109         61            0        100        2         98   
amo-ijca           167          3            3         97        5         95   
amo-minf-lazy      169          1            0        100        4         96   
amo-nomi-lazy      167          3            0        100        4         96   

bench                WGC      WOD                            Total  \
status        outof time complete outof memory outof time complete   
solver                                                               
clingo               300        0           83         17      111   
amo-ijca             300       24           76          0      196   
amo-minf-lazy        300       50           45          5      223   
amo-nomi-lazy        300       52           45          3      223   

bench                                  
status        outof memory outof time  
solver                                 
clingo                  83        576  
amo-ijca                79        495  
amo-minf-lazy           45        502  
amo-nomi-lazy           45        502